# Week 4 Exercise: Multi-Language Benchmark Arena

## By Mougang Thomas Gasmyr from the Wakanda Team

A competitive code conversion tool that takes Python code and simultaneously converts it to **Java**, **Rust**, and **TypeScript** using an LLM, then compiles, executes, and benchmarks each language.

### Features
- Parallel code generation via 3 simultaneous LLM calls (Option 2 strategy)
- Parallel compilation and execution across all 3 target languages
- Automatic performance benchmarking against the original Python
- Ranked leaderboard with speedup factors
- Graceful error handling — if one language fails, the others still compete
- Interactive Gradio UI with tabbed code views

## Setup

### Required
- At least one LLM API key (e.g. `OPENAI_API_KEY`)
- Java JDK (`javac` + `java`)
- Rust compiler (`rustc`)
- Node.js with `npx tsx` for TypeScript execution

### Optional API Keys
- `ANTHROPIC_API_KEY` — Claude models
- `GOOGLE_API_KEY` — Gemini models
- `OPENROUTER_API_KEY` — OpenRouter models
- Ollama running locally for free open-source models

In [ ]:
!pip install -q openai gradio python-dotenv

In [ ]:
# Imports

import os
import io
import sys
import time
import subprocess
import tempfile
import platform
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Constants

LANGUAGES = {
    "Java": {"extension": ".java", "filename": "Main.java"},
    "Rust": {"extension": ".rs", "filename": "main.rs"},
    "TypeScript": {"extension": ".ts", "filename": "main.ts"},
}

MODELS = [
    "gpt-4o-mini",
    "gpt-4o",
    "claude-sonnet-4-5-20250929",
]

In [ ]:
# Environment & Client Setup

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

keys = {
    "OpenAI": openai_api_key,
    "Anthropic": anthropic_api_key,
    "Google": google_api_key,
    "OpenRouter": openrouter_api_key,
}

for name, key in keys.items():
    status = f"exists (begins {key[:6]}...)" if key else "not set"
    print(f"{name}: {status}")

In [ ]:
# Initialize API Clients

openai_client = OpenAI()

anthropic_client = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/"
) if anthropic_api_key else None

gemini_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
) if google_api_key else None

ollama_client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

openrouter_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
) if openrouter_api_key else None

In [ ]:
# Model-to-Client Mapping

MODEL_CLIENTS = {
    "gpt-4o-mini": openai_client,
    "gpt-4o": openai_client,
    "claude-sonnet-4-5-20250929": anthropic_client,
    "gemini-2.5-pro": gemini_client,
}

def get_client(model):
    """Return the API client for a given model name."""
    client = MODEL_CLIENTS.get(model)
    if client is None:
        raise ValueError(f"No client available for model '{model}'. Check your API keys.")
    return client

In [ ]:
# System Information

SYSTEM_INFO = f"""
OS: {platform.system()} {platform.release()}
Architecture: {platform.machine()}
Processor: {platform.processor()}
""".strip()

print(SYSTEM_INFO)

In [ ]:
# Compile & Run Commands Per Language

def get_commands(language, work_dir):
    """Return (compile_command, run_command) for a given language and working directory."""
    if language == "Java":
        return (
            ["javac", "-d", work_dir, os.path.join(work_dir, "Main.java")],
            ["java", "-cp", work_dir, "Main"]
        )
    elif language == "Rust":
        output = os.path.join(work_dir, "main_rs")
        return (
            ["rustc", os.path.join(work_dir, "main.rs"),
             "-C", "opt-level=3", "-C", "target-cpu=native",
             "-C", "codegen-units=1", "-C", "lto=fat",
             "-C", "panic=abort", "-o", output],
            [output]
        )
    elif language == "TypeScript":
        return (
            None,
            ["npx", "tsx", os.path.join(work_dir, "main.ts")]
        )
    else:
        raise ValueError(f"Unsupported language: {language}")

In [ ]:
# Prompt Builders

def system_prompt_for(language):
    """Build the system prompt for a target language."""
    return (
        f"Your task is to convert Python code into high performance {language} code.\n"
        f"Respond only with {language} code. Do not provide any explanation other than occasional comments.\n"
        f"The {language} response needs to produce an identical output in the fastest possible time."
    )


def user_prompt_for(language, python_code):
    """Build the user prompt with system info and compile instructions."""
    lang_info = LANGUAGES[language]
    filename = lang_info["filename"]

    compile_note = ""
    if language == "Java":
        compile_note = "The class must be named Main. It will be compiled with: javac Main.java"
    elif language == "Rust":
        compile_note = "It will be compiled with: rustc main.rs -C opt-level=3 -C target-cpu=native"
    elif language == "TypeScript":
        compile_note = "It will be executed with: npx tsx main.ts (Node.js runtime)"

    return (
        f"Port this Python code to {language} with the fastest possible implementation "
        f"that produces identical output in the least time.\n\n"
        f"System information:\n{SYSTEM_INFO}\n\n"
        f"{compile_note}\n\n"
        f"Your response will be written to a file called {filename}.\n"
        f"Respond only with {language} code.\n\n"
        f"Python code to port:\n\n```python\n{python_code}\n```"
    )

In [ ]:
# Code Generation

import re

def extract_code(reply, language):
    """Extract clean code from an LLM response, stripping markdown fences and explanations."""
    # Try to extract from a fenced code block first
    lang_tags = {
        "Java": r"```(?:java)(.*?)```",
        "Rust": r"```(?:rust)(.*?)```",
        "TypeScript": r"```(?:typescript|ts)(.*?)```",
    }
    pattern = lang_tags.get(language, r"```(.*?)```")
    match = re.search(pattern, reply, re.DOTALL)
    if match:
        return match.group(1).strip()

    # Fallback: strip any remaining fences
    for fence in ["```java", "```rust", "```typescript", "```ts", "```"]:
        reply = reply.replace(fence, "")
    return reply.strip()


def generate_code(model, language, python_code):
    """Generate code for a single target language using the LLM."""
    client = get_client(model)
    messages = [
        {"role": "system", "content": system_prompt_for(language)},
        {"role": "user", "content": user_prompt_for(language, python_code)},
    ]
    response = client.chat.completions.create(model=model, messages=messages)
    reply = response.choices[0].message.content
    return extract_code(reply, language)

In [ ]:
# Parallel Code Generation

def generate_all_languages(model, python_code):
    """Generate code for all 3 languages in parallel."""
    results = {}
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(generate_code, model, lang, python_code): lang
            for lang in LANGUAGES
        }
        for future in as_completed(futures):
            lang = futures[future]
            try:
                results[lang] = future.result()
            except Exception as e:
                results[lang] = f"// Generation failed: {e}"
    return results

In [ ]:
# Python Runner

def run_python(code):
    """Execute Python code and return (output, execution_time)."""
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        start = time.time()
        exec(code, {"__builtins__": __builtins__})
        elapsed = time.time() - start
        output = buffer.getvalue()
    except Exception as e:
        elapsed = 0.0
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output.strip(), elapsed

In [ ]:
# Compile & Run Target Language

def compile_and_run(language, code):
    """Write code to a temp directory, compile (if needed), run, and return (output, time, error)."""
    work_dir = tempfile.mkdtemp(prefix=f"arena_{language.lower()}_")
    filename = LANGUAGES[language]["filename"]
    filepath = os.path.join(work_dir, filename)

    with open(filepath, "w") as f:
        f.write(code)

    compile_cmd, run_cmd = get_commands(language, work_dir)

    try:
        # Compile step (if applicable)
        if compile_cmd:
            subprocess.run(
                compile_cmd, check=True, text=True,
                capture_output=True, timeout=60
            )

        # Run and measure time
        start = time.time()
        result = subprocess.run(
            run_cmd, check=True, text=True,
            capture_output=True, timeout=120
        )
        elapsed = time.time() - start

        return result.stdout.strip(), elapsed, None

    except subprocess.CalledProcessError as e:
        return "", 0.0, f"Compilation/Runtime error:\n{e.stderr}"
    except subprocess.TimeoutExpired:
        return "", 0.0, "Execution timed out (>120s)"

In [ ]:
# Benchmark All Languages

def run_benchmark(python_code, generated_codes):
    """Run Python + all 3 languages and collect benchmark results."""
    # Run Python baseline
    py_output, py_time = run_python(python_code)
    results = {
        "Python": {"output": py_output, "time": py_time, "error": None}
    }

    # Run all 3 target languages in parallel
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(compile_and_run, lang, code): lang
            for lang, code in generated_codes.items()
        }
        for future in as_completed(futures):
            lang = futures[future]
            output, elapsed, error = future.result()
            results[lang] = {"output": output, "time": elapsed, "error": error}

    return results

In [ ]:
# Leaderboard Formatter

MEDALS = ["1st", "2nd", "3rd", "4th"]

def format_leaderboard(results):
    """Format benchmark results into a ranked markdown leaderboard."""
    py_time = results["Python"]["time"]

    # Separate successful and failed entries
    successful = []
    failed = []
    for lang, data in results.items():
        if data["error"]:
            failed.append((lang, data["error"]))
        else:
            successful.append((lang, data["time"]))

    # Sort by execution time (fastest first)
    successful.sort(key=lambda x: x[1])

    # Build leaderboard
    lines = ["## Leaderboard\n"]
    lines.append("| Rank | Language | Time (s) | Speedup vs Python |")
    lines.append("|------|----------|----------|-------------------|")

    for i, (lang, t) in enumerate(successful):
        rank = MEDALS[i] if i < len(MEDALS) else f"{i+1}th"
        speedup = f"{py_time / t:,.1f}x" if t > 0 else "N/A"
        lines.append(f"| {rank} | **{lang}** | {t:.6f} | {speedup} |")

    # Append failures
    if failed:
        lines.append("\n### Failed")
        for lang, error in failed:
            lines.append(f"- **{lang}**: {error[:200]}")

    return "\n".join(lines)

In [ ]:
# Main Orchestrator

def arena(python_code, model):
    """Run the full arena pipeline: generate -> benchmark -> leaderboard."""

    # Step 1: Generate code for all 3 languages in parallel
    generated = generate_all_languages(model, python_code)
    java_code = generated.get("Java", "// Generation failed")
    rust_code = generated.get("Rust", "// Generation failed")
    ts_code = generated.get("TypeScript", "// Generation failed")

    # Step 2: Benchmark all languages
    results = run_benchmark(python_code, generated)

    # Step 3: Format outputs
    leaderboard = format_leaderboard(results)

    java_out = results["Java"]["error"] or results["Java"]["output"]
    rust_out = results["Rust"]["error"] or results["Rust"]["output"]
    ts_out = results["TypeScript"]["error"] or results["TypeScript"]["output"]
    py_out = results["Python"]["output"]

    return java_code, rust_code, ts_code, py_out, java_out, rust_out, ts_out, leaderboard

In [ ]:
# Sample Python Code

SAMPLE_1 = """import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= (1 / j)
        j = i * param1 + param2
        result += (1 / j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {end_time - start_time:.6f} seconds")
"""

SAMPLE_2 = """import time

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

n = 10000
initial_seed = 42
min_val = -10
max_val = 10

start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [ ]:
# Gradio UI

with gr.Blocks(theme=gr.themes.Monochrome(), title="Multi-Language Benchmark Arena") as ui:

    gr.Markdown("# Multi-Language Benchmark Arena")
    gr.Markdown("Paste Python code, pick a model, and watch **Java**, **Rust**, and **TypeScript** compete.")

    # Input section
    with gr.Row():
        with gr.Column(scale=5):
            python_input = gr.Code(
                label="Python Code",
                value=SAMPLE_1,
                language="python",
                lines=20
            )
        with gr.Column(scale=1):
            model_dropdown = gr.Dropdown(
                choices=MODELS,
                value=MODELS[0],
                label="LLM Model"
            )
            run_btn = gr.Button("Run Arena", variant="primary")

    # Clickable sample buttons
    gr.Examples(
        examples=[[SAMPLE_1], [SAMPLE_2]],
        inputs=[python_input],
        label="Sample Programs",
        example_labels=["Sample 1: Pi Calculator", "Sample 2: Max Subarray Sum"],
    )

    # Generated code tabs
    with gr.Tabs():
        with gr.TabItem("Java"):
            java_code = gr.Code(label="Generated Java", language="c", lines=18)
            java_output = gr.TextArea(label="Java Output", lines=4)
        with gr.TabItem("Rust"):
            rust_code = gr.Code(label="Generated Rust", language="cpp", lines=18)
            rust_output = gr.TextArea(label="Rust Output", lines=4)
        with gr.TabItem("TypeScript"):
            ts_code = gr.Code(label="Generated TypeScript", language="typescript", lines=18)
            ts_output = gr.TextArea(label="TypeScript Output", lines=4)

    # Python output and leaderboard
    with gr.Row():
        with gr.Column(scale=1):
            python_output = gr.TextArea(label="Python Output (baseline)", lines=4)
        with gr.Column(scale=1):
            leaderboard = gr.Markdown(label="Leaderboard")

    # Wire the button
    run_btn.click(
        fn=arena,
        inputs=[python_input, model_dropdown],
        outputs=[java_code, rust_code, ts_code, python_output, java_output, rust_output, ts_output, leaderboard]
    )

In [ ]:
# Launch

ui.launch(inbrowser=True)